In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

!pip install -q transformers scikit-learn tqdm

In [1]:
# ============================================
# MODÈLE SVM POUR LA COMPARAISON AVEC BERT
# ============================================

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🚀 MODÈLE SVM - EXTRACTION DE MOTS-CLÉS")
print("="*60)

# === 1. CHARGEMENT DES DONNÉES ===
print("\n📂 Chargement des données pour SVM...")

# Utiliser les mêmes chemins que pour BERT
TRAIN_PATH = "/kaggle/input/datasets/duodou/data-bert/final_dataset_Train (1).csv"
VAL_PATH = "/kaggle/input/datasets/duodou/data-bert/validation_clean.csv"
TEST_PATH = "/kaggle/input/datasets/duodou/data-bert/testfin_clean.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Fonction safe_eval
def safe_eval(x):
    try:
        if isinstance(x, str):
            return eval(x)
        return x
    except:
        return []

# Convertir les keywords en listes
train_df["keywords_list"] = train_df["keywords"].apply(safe_eval)
val_df["keywords_list"] = val_df["keywords"].apply(safe_eval)
test_df["keywords_list"] = test_df["keywords"].apply(safe_eval)

# === 2. PRÉPARATION DES DONNÉES POUR SVM ===
# Pour SVM, nous devons transformer :
# - Les phrases en features (TF-IDF)
# - Les mots-clés en labels (multi-label classification)

print("\n📊 Préparation des données pour SVM...")

# Extraction des features TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.8,
    stop_words='english'
)

# Entraîner TF-IDF sur toutes les phrases
all_sentences = pd.concat([train_df["sentence"], val_df["sentence"], test_df["sentence"]])
tfidf.fit(all_sentences)

# Transformer les phrases en features
X_train = tfidf.transform(train_df["sentence"])
X_val = tfidf.transform(val_df["sentence"])
X_test = tfidf.transform(test_df["sentence"])

print(f"Features shape: {X_train.shape[1]} dimensions")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

# === 3. CRÉATION DES LABELS POUR LA CLASSIFICATION ===
# Collecter tous les mots-clés uniques
all_keywords = set()
for kw_list in tqdm(train_df["keywords_list"], desc="Collecting keywords"):
    all_keywords.update(kw_list)

all_keywords = sorted(list(all_keywords))
print(f"Nombre total de mots-clés uniques: {len(all_keywords)}")

# Créer un dictionnaire mot-clé -> index
keyword_to_idx = {kw: i for i, kw in enumerate(all_keywords)}

# Créer les labels binaires pour chaque phrase
def create_labels(keywords_list, keyword_to_idx):
    labels = np.zeros(len(keyword_to_idx))
    for kw in keywords_list:
        if kw in keyword_to_idx:
            labels[keyword_to_idx[kw]] = 1
    return labels

print("\n🏷️ Création des labels...")
y_train = np.array([create_labels(kw_list, keyword_to_idx) for kw_list in tqdm(train_df["keywords_list"], desc="Train labels")])
y_val = np.array([create_labels(kw_list, keyword_to_idx) for kw_list in tqdm(val_df["keywords_list"], desc="Val labels")])
y_test = np.array([create_labels(kw_list, keyword_to_idx) for kw_list in tqdm(test_df["keywords_list"], desc="Test labels")])

print(f"y_train shape: {y_train.shape}")
print(f"Proportion de mots-clés positifs: {y_train.sum() / y_train.size:.4f}")

# === 4. ENTRAÎNEMENT DU MODÈLE SVM ===
# Utiliser LinearSVC pour la classification multi-label avec OneVsRest
from sklearn.multiclass import OneVsRestClassifier
from sklearn.calibration import CalibratedClassifierCV

print("\n🔧 Entraînement du modèle SVM (OneVsRest)...")

# Paramètres optimisés pour SVM
svm_model = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=5000,
        random_state=42,
        dual='auto'
    ),
    n_jobs=-1
)

# Entraînement
svm_model.fit(X_train, y_train)
print("✅ Modèle SVM entraîné avec succès!")

# === 5. ÉVALUATION DU MODÈLE SVM ===
print("\n" + "="*50)
print("🎯 ÉVALUATION SVM SUR VALIDATION")
print("="*50)

# Prédictions sur validation
y_val_pred = svm_model.predict(X_val)

# Calcul des métriques au niveau token (toutes les classes)
val_accuracy = accuracy_score(y_val.flatten(), y_val_pred.flatten())
val_f1_micro = f1_score(y_val, y_val_pred, average='micro')
val_f1_macro = f1_score(y_val, y_val_pred, average='macro')
val_f1_weighted = f1_score(y_val, y_val_pred, average='weighted')

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation F1 (micro): {val_f1_micro:.4f}")
print(f"Validation F1 (macro): {val_f1_macro:.4f}")
print(f"Validation F1 (weighted): {val_f1_weighted:.4f}")

# Métriques par classe (moyenne binaire pour comparaison avec BERT)
# Pour comparaison équitable, on regarde la performance moyenne sur toutes les classes
print(f"\n📊 Performance moyenne par mot-clé:")
print(f"   F1-score moyen: {val_f1_macro:.4f}")

# === 6. ÉVALUATION SUR TEST ===
print("\n" + "="*50)
print("🎯 ÉVALUATION SVM SUR TEST")
print("="*50)

y_test_pred = svm_model.predict(X_test)

test_accuracy = accuracy_score(y_test.flatten(), y_test_pred.flatten())
test_f1_micro = f1_score(y_test, y_test_pred, average='micro')
test_f1_macro = f1_score(y_test, y_test_pred, average='macro')
test_f1_weighted = f1_score(y_test, y_test_pred, average='weighted')

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1 (micro): {test_f1_micro:.4f}")
print(f"Test F1 (macro): {test_f1_macro:.4f}")
print(f"Test F1 (weighted): {test_f1_weighted:.4f}")

# === 7. COMPARAISON AVEC BERT ===
print("\n" + "="*60)
print("📊 COMPARAISON BERT vs SVM")
print("="*60)

# Résultats BERT (d'après l'exécution précédente)
bert_test_accuracy = 0.9501
bert_test_f1 = 0.8546

print("\n" + "-"*50)
print("Comparaison sur l'ensemble de TEST:")
print("-"*50)
print(f"{'Métrique':<20} {'BERT':<15} {'SVM':<15} {'Différence':<15}")
print("-"*50)
print(f"{'Accuracy':<20} {bert_test_accuracy:<15.4f} {test_accuracy:<15.4f} {test_accuracy - bert_test_accuracy:+15.4f}")
print(f"{'F1-Score':<20} {bert_test_f1:<15.4f} {test_f1_macro:<15.4f} {test_f1_macro - bert_test_f1:+15.4f}")

# Interprétation
print("\n" + "-"*50)
print("📝 INTERPRÉTATION:")
print("-"*50)

if test_f1_macro > bert_test_f1:
    print("✅ SVM surpasse BERT sur ce jeu de données!")
    print(f"   Gain F1: +{(test_f1_macro - bert_test_f1)*100:.2f} points")
elif test_f1_macro < bert_test_f1:
    print("✅ BERT surpasse SVM sur ce jeu de données!")
    print(f"   Écart F1: {abs(test_f1_macro - bert_test_f1)*100:.2f} points en faveur de BERT")
else:
    print("✅ Les deux modèles ont des performances équivalentes")

print("\n📌 Notes:")
print("   - SVM est plus rapide à entraîner et plus léger")
print("   - BERT capture mieux le contexte des phrases")
print("   - Le choix dépend du besoin: rapidité vs précision contextuelle")

# === 8. FONCTION D'EXTRACTION POUR SVM ===
print("\n" + "="*50)
print("🧪 TEST EXTRACTEUR SVM")
print("="*50)

def extract_keywords_svm(text, tfidf_vectorizer, svm_model, keyword_list, threshold=0.5):
    """
    Extrait les mots-clés d'un texte avec SVM
    """
    if not text or not isinstance(text, str):
        return []
    
    # Transformer le texte en features
    X = tfidf_vectorizer.transform([text])
    
    # Prédire les probabilités (pour LinearSVC, on utilise decision_function)
    try:
        scores = svm_model.decision_function(X)[0]
        # Convertir en probabilités approximatives
        probs = 1 / (1 + np.exp(-scores))
    except:
        # Fallback pour d'autres types de SVM
        predictions = svm_model.predict(X)[0]
        probs = predictions.astype(float)
    
    # Sélectionner les mots-clés avec score > threshold
    keywords = [keyword_list[i] for i, prob in enumerate(probs) if prob > threshold]
    
    return keywords

# Tester sur les mêmes phrases que BERT
test_phrases = [
    "Machine learning improves healthcare.",
    "Artificial intelligence and deep learning are revolutionizing technology."
]

print("\nRésultats SVM:")
for phrase in test_phrases:
    kws = extract_keywords_svm(phrase, tfidf, svm_model, all_keywords, threshold=0.5)
    print(f"\nTexte: {phrase}")
    print(f"Mots-clés SVM: {kws}")

# === 9. ANALYSE SUPPLÉMENTAIRE ===
print("\n" + "="*50)
print("📈 ANALYSE SUPPLÉMENTAIRE")
print("="*50)

# Top 10 des mots-clés les plus fréquents
keyword_frequencies = y_train.sum(axis=0)
top_indices = np.argsort(keyword_frequencies)[-10:][::-1]
top_keywords = [all_keywords[i] for i in top_indices]
top_freqs = [keyword_frequencies[i] / len(train_df) * 100 for i in top_indices]

print("\nTop 10 mots-clés les plus fréquents:")
for kw, freq in zip(top_keywords, top_freqs):
    print(f"   {kw}: {freq:.2f}%")

# Matrice de confusion simplifiée (pour échantillon)
print("\n📊 Matrice de confusion sur test (échantillon):")
# Prendre un sous-ensemble pour éviter les OOM
sample_size = min(1000, len(test_df))
sample_indices = np.random.choice(len(test_df), sample_size, replace=False)
y_test_sample = y_test[sample_indices]
y_test_pred_sample = y_test_pred[sample_indices]

total_predictions = y_test_pred_sample.sum()
total_actual = y_test_sample.sum()
correct = (y_test_sample == y_test_pred_sample).sum()
print(f"   Total prédictions positives: {total_predictions}")
print(f"   Total réels positifs: {total_actual}")
print(f"   Correspondances exactes: {correct}")

print("\n✅ Analyse SVM terminée!")

🚀 MODÈLE SVM - EXTRACTION DE MOTS-CLÉS

📂 Chargement des données pour SVM...
Train: 12673 | Val: 1453 | Test: 1456

📊 Préparation des données pour SVM...
Features shape: 5000 dimensions
X_train: (12673, 5000) | X_val: (1453, 5000) | X_test: (1456, 5000)


Nombre total de mots-clés uniques: 20666

🏷️ Création des labels...


Test labels: 100%|██████████| 1456/1456 [00:00<00:00, 9335.57it/s]


y_train shape: (12673, 20666)
Proportion de mots-clés positifs: 0.0002

🔧 Entraînement du modèle SVM (OneVsRest)...
✅ Modèle SVM entraîné avec succès!

🎯 ÉVALUATION SVM SUR VALIDATION
Validation Accuracy: 0.9998
Validation F1 (micro): 0.5593
Validation F1 (macro): 0.0771
Validation F1 (weighted): 0.5000

📊 Performance moyenne par mot-clé:
   F1-score moyen: 0.0771

🎯 ÉVALUATION SVM SUR TEST
Test Accuracy: 0.9998
Test F1 (micro): 0.5595
Test F1 (macro): 0.0771
Test F1 (weighted): 0.4999

📊 COMPARAISON BERT vs SVM

--------------------------------------------------
Comparaison sur l'ensemble de TEST:
--------------------------------------------------
Métrique             BERT            SVM             Différence     
--------------------------------------------------
Accuracy             0.9501          0.9998                  +0.0497
F1-Score             0.8546          0.0771                  -0.7775

--------------------------------------------------
📝 INTERPRÉTATION:
---------------